In [ ]:
# Prefect Setup
# !pip install prefect
import warnings; warnings.filterwarnings('ignore')
try:
    import prefect
    print(f'Prefect {prefect.__version__} ready!')
except ImportError:
    print('Run: pip install prefect')

# 🔧 Flows, Tasks & Task Configuration in Prefect

---

## 🤔 Flows vs Tasks — When to Use Which?

- **`@flow`**: The top-level pipeline. Calls tasks in order. Can call other flows (subflows).
- **`@task`**: A single unit of work — one API call, one transformation, one file write.

> 💡 **Rule**: If it can fail independently and be retried, it should be a `@task`. If it orchestrates multiple tasks, it should be a `@flow`.

---

## 💻 Example 1: Task Configuration Options

In [ ]:
from prefect import task, flow
from datetime import timedelta
import requests, pandas as pd

@task(
    name         = "Fetch Sales from API",     # Display name in UI
    description  = "Calls the sales REST API and returns JSON data",
    retries      = 3,                          # Retry 3 times on failure
    retry_delay_seconds = 30,                  # Wait 30s between retries
    timeout_seconds     = 120,                 # Fail if takes > 2 minutes
    tags         = ["api", "extract"],         # Group/filter in UI
    log_prints   = True,                       # Capture print() as logs
)
def fetch_sales_api(date: str) -> list:
    """Fetch sales records for a given date from the REST API."""
    url = f"https://api.company.com/sales?date={date}"
    response = requests.get(url, timeout=30)
    response.raise_for_status()   # Raises error on 4xx/5xx → triggers retry!
    data = response.json()
    print(f"Fetched {len(data):,} records for {date}")
    return data

@task(name="Validate Data", retries=0)
def validate_data(records: list) -> list:
    """Remove records with missing order_id or negative amounts."""
    valid = [r for r in records if r.get("order_id") and r.get("amount", 0) > 0]
    print(f"Valid: {len(valid):,} / Total: {len(records):,}")
    return valid

@flow(name="Sales ETL Pipeline", log_prints=True)
def sales_etl(date: str = "2024-01-15"):
    records   = fetch_sales_api(date)
    validated = validate_data(records)
    print(f"Pipeline complete for {date}: {len(validated):,} clean records")
    return len(validated)

# Run locally
sales_etl("2024-01-15")

---

## 💻 Example 2: Task Dependencies & Parallel Execution

In [16]:
from prefect import flow, task
import time

@task
def extract_sales() -> str:
    time.sleep(1)
    return "sales_data"

@task
def extract_customers() -> str:
    time.sleep(1)       # Both run ~simultaneously in parallel!
    return "customer_data"

@task
def extract_products() -> str:
    time.sleep(1)
    return "product_data"

@task
def join_and_load(sales, customers, products) -> None:
    print(f"Joining: {sales} + {customers} + {products}")

@flow(name="Parallel Extract ETL")
def parallel_etl():
    # These 3 tasks submit immediately — Prefect runs them in parallel
    sales     = extract_sales.submit()      # .submit() = async, returns Future
    customers = extract_customers.submit()
    products  = extract_products.submit()

    # This task waits for all 3 to complete before running
    join_and_load(
        sales.result(),       # .result() blocks until the task is done
        customers.result(),
        products.result(),
    )

parallel_etl()   # Total time ~1s, not ~3s!
print("Use .submit() for parallel tasks, .result() to wait for output")

05:21:04.947 | INFO    | Flow run 'grinning-bug' - Beginning flow run 'grinning-bug' for flow 'Parallel Extract ETL'

05:21:04.949 | INFO    | Flow run 'grinning-bug' - View at https://app.prefect.cloud/account/4322fe39-684d-4b13-885d-09cce5a07ba0/workspace/1b94aff5-156b-4045-949a-1314ed297048/runs/flow-run/069c0806-7ee1-7e85-8000-076bed13fad9

05:21:05.957 | INFO    | Task run 'extract_sales-80e' - Finished in state Completed()

05:21:05.959 | INFO    | Task run 'extract_products-732' - Finished in state Completed()

05:21:05.960 | INFO    | Task run 'extract_customers-6d0' - Finished in state Completed()

Joining: sales_data + customer_data + product_data


05:21:05.964 | INFO    | Task run 'join_and_load-f08' - Finished in state Completed()

05:21:06.956 | INFO    | Flow run 'grinning-bug' - Finished in state Completed()

Use .submit() for parallel tasks, .result() to wait for output


---

## 💻 Example 3: Conditional Task Execution

In [8]:
from prefect import flow, task
from prefect.context import get_run_context

@task
def extract(source: str) -> list:
    print(f"Extracting from {source}")
    return [{"id": 1}, {"id": 2}]

@task
def validate(records: list) -> bool:
    is_valid = len(records) > 0
    print(f"Validation: {'passed' if is_valid else 'FAILED'}")
    return is_valid

@task
def load(records: list) -> None:
    print(f"Loading {len(records)} records")

@task
def send_alert(message: str) -> None:
    print(f"🚨 ALERT: {message}")

@flow(name="Conditional ETL")
def conditional_etl(source: str = "sales_api"):
    records = extract(source)
    is_valid = validate(records)

    if is_valid:
        load(records)
    else:
        send_alert(f"Validation failed for {source} — pipeline aborted!")

conditional_etl()

05:08:22.903 | INFO    | Flow run 'thoughtful-locust' - Beginning flow run 'thoughtful-locust' for flow 'Conditional ETL'

05:08:22.905 | INFO    | Flow run 'thoughtful-locust' - View at https://app.prefect.cloud/account/4322fe39-684d-4b13-885d-09cce5a07ba0/workspace/1b94aff5-156b-4045-949a-1314ed297048/runs/flow-run/069c07d6-dfdc-7400-8000-58dab46aa965

Extracting from sales_api


05:08:22.907 | INFO    | Task run 'extract-671' - Finished in state Completed()

Validation: passed


05:08:22.909 | INFO    | Task run 'validate-cb8' - Finished in state Completed()

Loading 2 records


05:08:22.912 | INFO    | Task run 'load-84b' - Finished in state Completed()

05:08:23.907 | INFO    | Flow run 'thoughtful-locust' - Finished in state Completed()

---

## 💻 Example 4: Task Caching — Skip Redundant Runs

In [ ]:
from prefect import task, flow
from prefect.tasks import task_input_hash
from datetime import timedelta
import json

@task(
    cache_key_fn     = task_input_hash,
    cache_expiration = timedelta(hours=1),
    name             = "Expensive API Call",
)
def fetch_exchange_rates(base_currency: str) -> dict:
    """Fetch exchange rates — cached so we don't call the API every minute."""
    import requests
    response = requests.get(f"https://api.frankfurter.app/latest?from={base_currency}")
    
    # API in JSON
    # print("Full API Response:", json.dumps(response.json(), indent=4))
    rates = response.json()["rates"]
    print(f"Fetched {len(rates)} exchange rates for {base_currency}")
    return rates

@flow
def etl_with_caching():
    rates1 = fetch_exchange_rates("USD")  # Calls API
    rates2 = fetch_exchange_rates("USD")  # Uses CACHE — no API call!
    rates3 = fetch_exchange_rates("EUR")  # Different input → calls API again
    print("Done")

etl_with_caching()
print("Caching = skip redundant task runs within the cache_expiration window")

05:17:32.078 | INFO    | Flow run 'economic-frigatebird' - Beginning flow run 'economic-frigatebird' for flow 'etl-with-caching'

05:17:32.079 | INFO    | Flow run 'economic-frigatebird' - View at https://app.prefect.cloud/account/4322fe39-684d-4b13-885d-09cce5a07ba0/workspace/1b94aff5-156b-4045-949a-1314ed297048/runs/flow-run/069c07f9-310e-77dd-8000-947823603b65

Fetched 29 exchange rates for USD


05:17:32.990 | INFO    | Task run 'Expensive API Call-40a' - Finished in state Completed()

05:17:32.993 | INFO    | Task run 'Expensive API Call-9bd' - Finished in state Cached(type=COMPLETED)

Fetched 29 exchange rates for EUR


05:17:33.034 | INFO    | Task run 'Expensive API Call-0c8' - Finished in state Completed()

Done


05:17:34.089 | INFO    | Flow run 'economic-frigatebird' - Finished in state Completed()

Caching = skip redundant task runs within the cache_expiration window


---

## ⚠️ Common Mistakes

In [15]:
mistakes = [
    ("Calling .result() before submitting all tasks",
     "Submit all tasks first (.submit()), then call .result() → enables parallelism"),
    ("No retries on API/network tasks",
     "APIs fail transiently — always add retries=3 for external calls"),
    ("Heavy computation in @flow body",
     "Put computation in @task functions — flows should only orchestrate"),
]
for mistake, fix in mistakes:
    print(f"❌ {mistake}")
    print(f"✅ Fix: {fix}\n")

❌ Calling .result() before submitting all tasks
✅ Fix: Submit all tasks first (.submit()), then call .result() → enables parallelism

❌ No retries on API/network tasks
✅ Fix: APIs fail transiently — always add retries=3 for external calls

❌ Heavy computation in @flow body
✅ Fix: Put computation in @task functions — flows should only orchestrate

